# RAG v2 — Alert-Triggered Grounded Explanations (revised evaluation)

Presents the v2 explanation pipeline results (paper §4.4–4.10). v2 changes vs v1:

- **Query semantics fixed**: z-scores vs each subject's *non-flagged* windows (the
  detector's normal population), evidence-tied character phrases, online-capable.
- **Corpus expanded** with wearable-relevant guidance (EHRA 2022 digital-devices
  guide; 2023 ACC/AHA AF guideline).
- **Raw pre-canonicalization text preserved** → citation accuracy reported before
  AND after repair (v1's "99.3% → 100%" was not reconstructable).
- **Judges validated** on a 200-item corruption benchmark before use; v1's judge
  (llama3.1:8b) detects 0/100 corruptions.
- **Labeled-event evaluation**: 148 true events (WESAD stress, MIT-BIH ectopy,
  PTB-XL pathology) with labels never entering the queries.
- Atomic-claim verification + released clinician kit.

Cold-start rebuild: this notebook delegates to `scripts/v2/rag_pipeline_v2.py`
and `scripts/v2/judge_eval_v2.py` when artifacts are missing (GPU generation
~2 h + judging ~1 h + OpenRouter key required for API columns).

In [1]:
import json, re
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd()
OUT = ROOT / "outputs_v2"
GEN = OUT / "generation_v2.jsonl"
rows = [json.loads(l) for l in open(GEN, encoding="utf-8")]
sup = json.loads((OUT / "superseded_keys.json").read_text())["superseded_mitbih_keys"]
rows = [r for r in rows if r["key"] not in sup]
df = pd.DataFrame([{k: r.get(k) for k in ("key", "group", "subgroup", "model", "prompt",
                                           "true_label", "latency_sec")} for r in rows])
df.groupby(["group", "subgroup"]).size().to_frame("n")

n
group  subgroup        
dalia  genablation   50
       main         398
       wordcap       50
mitbih labeled       50
ptbxl  labeled       48
wesad  labeled       50

## Corpus v2 (206 documents / 5,045 chunks)

In [2]:
cs = json.load(open(OUT / "corpus_stats_v2.json"))
print({k: cs[k] for k in ("docs_total", "chunks_total", "chunks_tier1_v1", "chunks_tier1_v2",
                          "chunks_tier2", "mean_chunk_words")})
pd.DataFrame([m for m in cs["tier1_v2_manifest"]])

{'docs_total': 206, 'chunks_total': 5045, 'chunks_tier1_v1': 679, 'chunks_tier1_v2': 313, 'chunks_tier2': 4053, 'mean_chunk_words': 489.1}


,slug,title,doi,status,provenance,license,words
0,ehra2022_digital_devices_arrhythmias,How to use digital devices to detect and manag...,10.1093/europace/euac038,included,PMC HTML https://pmc.ncbi.nlm.nih.gov/articles...,free-to-read (non-OA deposit),20524
1,accaha2023_af_guideline,2023 ACC/AHA/ACCP/HRS Guideline for the Diagno...,10.1016/j.jacc.2023.08.017,included,NCBI-efetch PMC11104284,open-access full text (API),119730
2,esc2024_af_guideline,2024 ESC Guidelines for the management of atri...,10.1093/eurheartj/ehae178,excluded,landing page: HTTPError,unknown,0


## Citation audit — raw vs repaired (paper §4.4)

In [3]:
ca = json.load(open(OUT / "citation_audit_v2.json"))
print(json.dumps(ca, indent=2)[:900])

{
  "n_rows": 398,
  "raw": {
    "citations": 1208,
    "valid": 1196,
    "rows_all_valid": 389,
    "accuracy": 0.9901
  },
  "repaired": {
    "citations": 1196,
    "valid": 1196,
    "rows_all_valid": 398,
    "accuracy": 1.0
  },
  "snaps": 0,
  "drops": 12,
  "tier1_name_citations": 109,
  "tier1_name_unmatched": 8
}


## Judge validation on the corruption benchmark (paper §4.5)

In [4]:
jv = json.load(open(OUT / "judge_validation.json"))
pd.DataFrame(jv["per_judge"]).T
print("selected local judge:", jv["selected_local_judge"]["model"],
      "| criterion:", jv["selected_local_judge"]["criterion"])

selected local judge: gemma4:e4b | criterion: detection >= 0.3 on corruption benchmark, then max (detection - FP)


## Main judge scores (validated local judge; paper §4.6)

In [5]:
ev = pd.read_csv(OUT / "rag_evaluation_v2.csv")
agg = ev.groupby("subgroup").agg(n=("local_faithfulness", "size"),
                                 faith=("local_faithfulness", "mean"),
                                 relev=("local_relevance", "mean"),
                                 compl=("local_completeness", "mean")).round(2)
agg
m = ev[(ev.group == "dalia") & (ev.subgroup == "main")]
print("dalia faithfulness dist:", m.local_faithfulness.value_counts().sort_index().to_dict())

dalia faithfulness dist: {0: 45, 1: 2, 2: 176, 3: 175}


## Labeled-event concordance — the integration result (paper §4.7)

In [6]:
cc = json.load(open(OUT / "concordance_v2.json"))
tab = pd.DataFrame({g: {"n": v["n"], "concordant": v["concordant"],
                        "concordant_%": round(100 * v["concordant"] / v["n"]),
                        "artifact_language": v["artifact_conclusion"]}
                    for g, v in cc.items()}).T
tab

,n,concordant,concordant_%,artifact_language
wesad,50,47,94,28
mitbih,50,6,12,28
ptbxl,48,3,6,20


## Before/after query-and-corpus fix (398 wearable alerts, same flags)

In [7]:
nd2 = json.load(open(OUT / "near_duplicate_v2.json"))
nd1 = json.load(open(OUT / "rag_analysis_v1" / "near_duplicate_summary.json"))
pd.DataFrame({"v1": {"clusters@0.9": nd1["n_clusters_at_0.9"], "mean_NN_cos": nd1["mean_nn_cos"],
                     "pct_twins>0.9": nd1["pct_rows_with_nn_gt_0.9"], "guideline_reach_%": 6.5},
              "v2": {"clusters@0.9": nd2["n_clusters_at_0.9"], "mean_NN_cos": nd2["mean_nn_cos"],
                     "pct_twins>0.9": nd2["pct_rows_with_nn_gt_0.9"],
                     "guideline_reach_%": round(100 * nd2["tier1_alert_rate"], 1)}}).T

,clusters@0.9,mean_NN_cos,pct_twins>0.9,guideline_reach_%
v1,173.0,0.9356,81.66,6.5
v2,237.0,0.9149,67.59,17.6


## Atomic-claim verification (paper §4.9)

In [8]:
fs = json.load(open(OUT / "factscore_lite.json"))
print(json.dumps(fs, indent=2))

{
  "n_explanations": 60,
  "n_claims": 797,
  "pct_supported": 52.32,
  "pct_unsupported": 0.0,
  "pct_unverifiable": 47.68,
  "verifier": "gemma4:e4b (different family from generator)",
  "by_group": {
    "dalia": {
      "n_claims": 414,
      "pct_supported": 47.34
    },
    "mitbih": {
      "n_claims": 103,
      "pct_supported": 67.96
    },
    "wesad": {
      "n_claims": 151,
      "pct_supported": 54.3
    },
    "ptbxl": {
      "n_claims": 129,
      "pct_supported": 53.49
    }
  }
}


## Ablations (paper §4.8) + latency

In [9]:
for sg in ("main", "wordcap", "genablation"):
    s = ev[(ev.subgroup == sg) & (ev.group == "dalia")] if sg == "main" else ev[ev.subgroup == sg]
    words = np.mean([len(r["explanation"].split()) for r in rows
                     if (r["subgroup"] == sg and (sg != "main" or r["group"] == "dalia"))])
    print(f"{sg:12s} n={len(s):3d} words={words:6.1f} faith={s.local_faithfulness.mean():.2f} compl={s.local_completeness.mean():.2f}")
mn = [r for r in rows if r["group"] == "dalia" and r["subgroup"] == "main"]
print(f"generation latency: mean {np.mean([r['latency_sec'] for r in mn]):.1f} s/alert")

main         n=398 words= 127.3 faith=2.21 compl=2.20
wordcap      n= 50 words= 196.5 faith=2.02 compl=2.12
genablation  n= 50 words=  99.1 faith=1.78 compl=1.52
generation latency: mean 11.0 s/alert


## Example alert (paper §4.10)

In [10]:
def sec(t, a, b):
    m = re.search(rf"{a}:\s*(.*?)(?={b}:|$)", t, re.S)
    return m.group(1).strip() if m else ""

ex = next(r for r in rows if r["group"] == "wesad"
          and "stress" in sec(r["explanation"], "DETECTED", "EVIDENCE").lower())
print("TRUE LABEL:", ex["true_label"])
print("QUERY:", ex["query"])
print()
print(ex["explanation"])

TRUE LABEL: stress
QUERY: Biosignal window flagged by anomaly detection. reduced bvp (z=-7.2 vs subject baseline) elevated wrist_eda (z=+4.0 vs subject baseline). Relevant topics: electrodermal activity skin conductance sympathetic arousal stress physiology photoplethysmography heart rate variability ECG. Other readings: ecg mean=0.00, resp mean=0.55, wrist_temp mean=32.23.

DETECTED: The pattern suggests acute sympathetic arousal (stress) where reduced blood volume pulse and elevated skin conductance indicate a physiological stress response common in anxiety or high cognitive load scenarios .

EVIDENCE: EDA reflects changes from sweat gland activity modulated by the autonomic nervous system, making it valuable for detecting emotional arousal and stress . When stressed, blood pressure increases causing higher heart rate linked with low HRV (reduced BVP) . EDA tends to increase during stressful periods while adding noise can affect PPG signals like the reduced BVP seen here .

RECOMMEND

## Reading

Under a validated judge the system's faithfulness averages 2.2/3 with ~44% of
explanations containing extrapolated claims; 47.7% of atomic claims are
unverifiable from the retrieved context; and explanations name the true
condition for 94% of stress events but 12% / 6% of ECG pathology, attributing
42–56% of true pathology to artifact. The clinician kit in `clinician_eval/`
adjudicates these findings with human raters.